In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [6]:
!pip install -q datasets transformers sentence-transformers

import torch
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

dataset = load_dataset(
    "csv",
    data_files=train_path,
    split="train"
)

def add_combined_text(row):
    row["combined_text"] = row["prompt"] + " " + row["A"]
    return row

dataset = dataset.map(add_combined_text)

print("Length:", len(dataset[51]["combined_text"]))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Length: 614


In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Vocabulary size:", tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 30522


In [9]:
print("SEP token ID:", tokenizer.sep_token_id)

SEP token ID: 102


In [10]:
prompts = list(dataset["prompt"])

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Shape:", encoded["input_ids"].shape)

Shape: torch.Size([2000, 128])


In [11]:
hidden_size = 768
attention_heads = 12

head_dimension = hidden_size // attention_heads

print("Head dimension:", head_dimension)

Head dimension: 64


In [12]:
model = AutoModel.from_pretrained("bert-base-uncased")

prompt = dataset[0]["prompt"]

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

print("Shape:", outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Shape: torch.Size([1, 31, 768])


In [13]:
cls_vector = outputs.last_hidden_state[0, 0]

first_five = cls_vector[:5]

print("First 5 values:", first_five)
print("Sum:", round(first_five.sum().item(), 4))

First 5 values: tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480])
Sum: -1.2001


In [14]:
attention_model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True,
    attn_implementation="eager"
)

text = "Light-ion fusion is a technique."

attention_inputs = tokenizer(
    text,
    return_tensors="pt"
)

print(tokenizer.convert_ids_to_tokens(attention_inputs["input_ids"][0]))

with torch.no_grad():
    attention_outputs = attention_model(**attention_inputs)

last_layer = attention_outputs.attentions[-1]

first_head = last_layer[0, 0]

tokens = tokenizer.convert_ids_to_tokens(
    attention_inputs["input_ids"][0]
)

fusion_index = tokens.index("fusion")

attention_weight = first_head[0, fusion_index].item()

print("Fusion index:", fusion_index)
print("Attention weight:", round(attention_weight, 4))

['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Fusion index: 4
Attention weight: 0.1025


In [15]:
sentence_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

prompt_embedding = sentence_model.encode(prompt)
option_embedding = sentence_model.encode(option_b)

similarity = util.cos_sim(
    prompt_embedding,
    option_embedding
).item()

print("Similarity:", round(similarity, 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity: 0.7658


In [16]:
def map_at_3(actual, predictions):
    predictions = predictions[:3]

    if actual not in predictions:
        return 0.0

    rank = predictions.index(actual) + 1

    return 1 / rank

In [17]:
options = ["A", "B", "C", "D", "E"]

all_text = []

for row in dataset:
    all_text.append(row["prompt"])

    for option in options:
        all_text.append(row[option])

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(all_text)

tfidf_predictions = []

for row in dataset:

    prompt_vector = tfidf.transform([row["prompt"]])

    scores = {}

    for option in options:
        option_vector = tfidf.transform([row[option]])

        scores[option] = cosine_similarity(
            prompt_vector,
            option_vector
        )[0][0]

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    tfidf_predictions.append(ranked[:3])

In [18]:
minilm_predictions = []
minilm_scores = []

for row in dataset:

    texts = [row["prompt"]] + [row[x] for x in options]

    embeddings = sentence_model.encode(
        texts,
        convert_to_tensor=True
    )

    prompt_embedding = embeddings[0]
    option_embeddings = embeddings[1:]

    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0]

    ranking = torch.argsort(
        similarities,
        descending=True
    )

    top3 = [
        options[i.item()]
        for i in ranking[:3]
    ]

    minilm_predictions.append(top3)

    minilm_scores.append(
        map_at_3(row["answer"], top3)
    )

In [19]:
minilm_map3 = np.mean(minilm_scores)

improved = 0

for i, row in enumerate(dataset):

    answer = row["answer"]

    if (
        answer not in tfidf_predictions[i]
        and answer in minilm_predictions[i]
    ):
        improved += 1

print("MiniLM MAP@3:", round(minilm_map3, 2))
print("Improved:", improved)

MiniLM MAP@3: 0.42
Improved: 462


In [20]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

row = dataset[1]

candidate_labels = [
    row["A"],
    row["B"],
    row["C"]
]

result = zero_shot(
    row["prompt"],
    candidate_labels=candidate_labels
)

print(result["labels"])
print(result["scores"])

print("Top score:", round(result["scores"][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement an

In [21]:
normal_result = zero_shot(
    row["prompt"],
    candidate_labels=candidate_labels
)

multi_result = zero_shot(
    row["prompt"],
    candidate_labels=candidate_labels,
    multi_label=True
)

normal_sum = sum(normal_result["scores"])
multi_sum = sum(multi_result["scores"])

difference = abs(normal_sum - multi_sum)

print("Normal sum:", normal_sum)
print("Multi-label sum:", multi_sum)
print("Difference:", round(difference, 4))

Normal sum: 0.9999999701976776
Multi-label sum: 0.0005095975611766335
Difference: 0.9995


In [22]:
!pip install -q "transformers==4.57.6" sentencepiece

In [23]:
from datasets import load_dataset

train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

dataset = load_dataset(
    "csv",
    data_files=train_path,
    split="train"
)

print(dataset)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


In [24]:
import torch
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device=0 if torch.cuda.is_available() else -1
)

row = dataset[0]

input_text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

result = generator(
    input_text,
    max_new_tokens=5
)

print(result[0]["generated_text"])

Device set to use cuda:0


B
